In [1]:
%load_ext autoreload
%autoreload 2

# Preparation of public databes for sMOMENT/workflow

Our workflows depend on the autoPACMEN library and use its modules directly in our scripts. Furthrmore, we forked the autoPACMEN library and made some adaptions, e.g., deletion of cached files directory because this can cause some trouble with windows file naming convention and changed url for calling the SBAIO-RK API. Therefore, before running this script, please clone the repo from our fork (https://github.com/voidsailor/autopacmen), check out the tag "sMOMNT_with_MitoCore" and enter the path to the cloned repo below.

This script executes all sMOMENT functions to prepare information from BRENDA, Sabio-RK, and BiGG for the sMOMENT workflow. Execute before running the workflow. Update whenever model or databases changed!

In [2]:
# clone autopacmen repository beforehand
import sys
# enter path to the cloned autopacmen repo here
sys.path.append('C:/Users/emanuel.lange/git/autopacmen')

In [ ]:
# database preparation
from autopacmen.submodules.create_combined_kcat_database import create_combined_kcat_database
from utilities.parse_brenda_textfile import parse_brenda_textfile # we had to adapt the original function as it was not able to parse the newer version of the BRENDA text file
from autopacmen.submodules.parse_bigg_metabolites_file import parse_bigg_metabolites_file
from autopacmen.submodules.parse_brenda_json_for_model import parse_brenda_json_for_model
from autopacmen.submodules.parse_sabio_rk_for_model import parse_sabio_rk_for_model_with_sbml
from autopacmen.submodules.get_reactions_kcat_mapping import get_reactions_kcat_mapping

# Paths

In [ ]:
organism = 'homo sapiens'

bigg_metabolites_path = '../public_information/bigg_models_metabolites.txt'
bigg_metabolites_json_path = '../public_information'
bigg_metabolites_json = '../public_information/bigg_id_name_mapping.json' 

brenda_path = '../public_information/brenda_2024_1.txt'
brenda_json_path = '../public_information/brenda_with_bigg_metabolites.json'

parameterized_models_path_dict = {
    'Mitocore_Preliminary': './../parameterized_plt_models/Mitocore_Preliminary_plt.xml',
    'Mitocore_MitoMammal': './../parameterized_plt_models/Mitocore_MitoMammal_plt.xml',
    'Mitocore_aligned_to_Human1': './../parameterized_plt_models/Mitocore_aligned_to_Human1_plt.xml',
}

smoment_paths_dict = {}

for model_name in parameterized_models_path_dict.keys():
    smoment_paths_dict[model_name] = {
        "brenda_json_path": f"../public_information/{model_name}_brenda.json",
        "sabio_rk_json_path": f"../public_information/{model_name}_sabio_rk.json",
        "combined_database_path": f"../public_information/{model_name}_combined_database.json",
        "kcat_mapping_path": f"../public_information/{model_name}_reactions_kcat_mapping_combined.json"
    }

## 1.1) parsing bigg metabolites


In [5]:
parse_bigg_metabolites_file(bigg_metabolites_path, bigg_metabolites_json_path)

## 1.2) parsing BRENDA

Parsing of the BRENDA file has a sloppy implementation. I fixed it to work with BRENDA 2024_1 but it won't work with older versions because BRENDA changed the annotation of species with protein ids.

In [8]:
parse_brenda_textfile(brenda_path, bigg_metabolites_json_path, brenda_json_path)

## 2) Model specific BRENDA database

In [9]:
for model_name in parameterized_models_path_dict.keys():
    parse_brenda_json_for_model(parameterized_models_path_dict[model_name], brenda_json_path, smoment_paths_dict[model_name]["brenda_json_path"])

## 3) Sabio-RK

In [10]:
for model_name in parameterized_models_path_dict.keys():
    parse_sabio_rk_for_model_with_sbml(parameterized_models_path_dict[model_name], smoment_paths_dict[model_name]["sabio_rk_json_path"], bigg_metabolites_json)

Starting EC numbers kcat search in SABIO-RK...
Wildcard level 0...
['1.6.5.8', '1.1.1.35', '3.5.1.9', '1.2.1.12', '4.1.1.45', '5.2.1.2', '2.3.1.29', '6.3.1.2', '4.2.1.2', '4.1.3.4', '1.14.99.3', '1.3.3.3', '5.3.1.1', '4.2.1.3', '1.2.1.27', '3.3.1.1', '4.4.1.1', '2.3.1.39', '2.6.1.42', '1.4.3.14', '1.10.2.2', '3.5.4.6', '2.6.1.1', '4.1.1.41', '2.6.1.52', '1.15.1.1', '6.4.1.1', '4.2.1.11', '4.1.1.15', '2.7.3.2', '1.2.1.88', '6.2.1.2', '5.4.99.2', '2.8.1.2', '3.1.2.14', '1.13.11.11', '1.1.1.2', '5.3.1.9', '2.7.1.30', '2.6.1.13', '4.1.1.37', '3.5.3.1', '1.2.1.3', '6.3.2.3', '1.11.1.9', '2.7.1.40', '1.2.4.4', '2.3.1.85', '1.2.4.2', '3.5.1.63', '1.1.1.49', '2.7.1.2', '1.5.1.34', '1.5.1.2', '1.6.5.3', '6.3.5.4', '6.3.4.4', '1.1.1.28', '1.4.3.2', '2.6.1.2', '4.3.2.1', '1.1.1.41', '6.2.1.1', '2.1.2.10', '1.9.3.1', '2.7.8.5', '1.1.1.8', '4.99.1.1', '1.1.1.38', '6.4.1.2', '2.7.8.41', '1.3.8.8', '3.6.1.1', '1.3.8.4', '1.2.1.32', '5.1.3.1', '1.14.16.1', '2.7.7.41', '1.8.1.7', '2.7.4.6', '2.7.1.11',

## 4) combine model specific BRENDA and SabioRK

In [12]:
for model_name in parameterized_models_path_dict.keys():
    create_combined_kcat_database(smoment_paths_dict[model_name]["sabio_rk_json_path"], smoment_paths_dict[model_name]["brenda_json_path"], smoment_paths_dict[model_name]["combined_database_path"])

## 5) Map kcat values to EC numbers

In [13]:
for model_name in parameterized_models_path_dict.keys():
    get_reactions_kcat_mapping(parameterized_models_path_dict[model_name], bigg_metabolites_json_path, model_name, organism, smoment_paths_dict[model_name]["combined_database_path"], "", "median")

***
Reaction: EX_2hb_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_ac_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_acac_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_akg_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_ala_B_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_ala_L_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_arg_L_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_argsuc_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_asn_L_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_asp_L_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_bhb_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_bilirub_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_biomass_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_but_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_chol_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_cit_e
Forward kcat: nan
Reverse kcat: nan

***
Reaction: EX_c

# Determine number of reactions with kcat

In [14]:
import json
import cobra

def determine_total_kcats(reaction_kcat_mapping_path, model_path):

    # get reactions that dont have kcat
    with open(reaction_kcat_mapping_path) as file:
        kcat_dict = json.load(file)

    exclude_reactions = []

    for reaction_id, kcats in kcat_dict.items():
        if str(kcats['forward']) == 'nan':
            exclude_reactions.append(reaction_id)

    # for some reactions no kcat is available, exclude those
    model = cobra.io.read_sbml_model(model_path)

    for reaction in model.reactions:
        if reaction.id not in kcat_dict.keys():
            exclude_reactions.append(reaction.id)
            
    number_of_kcats = []

    for reaction_id, kcats in kcat_dict.items():
        if reaction_id in exclude_reactions:
            continue
        number_of_kcats.append(kcats['forward'])

        # if not kcats['forward'] == kcats['reverse']:
        #     number_of_kcats.append(kcats['reverse'])

    print(len(number_of_kcats), "reactions with kcat values")

In [15]:
for model_name in parameterized_models_path_dict.keys():
    print(model_name)
    determine_total_kcats(smoment_paths_dict[model_name]["kcat_mapping_path"], parameterized_models_path_dict[model_name])

Mitocore_Original
295 reactions with kcat values
Mitocore_Preliminary
295 reactions with kcat values
Mitocore_MitoMammal
298 reactions with kcat values
Mitocore_aligned_to_Human1
309 reactions with kcat values
